<a href="https://colab.research.google.com/github/yafiestiawan/menegement/blob/main/SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""Student Database System - Python + SQLite.

Bisa dijalankan di Google Colab atau Python lokal.
Di Colab, database secara default disimpan di Google Drive.
"""

from pathlib import Path
from contextlib import contextmanager
import sqlite3

# ============================================================
# 1. PENGATURAN LOKASI DATABASE
# ============================================================

try:
    from google.colab import drive
    DI_GOOGLE_COLAB = True
except ImportError:
    drive = None
    DI_GOOGLE_COLAB = False

# Ubah ke False jika di Google Colab ingin memakai penyimpanan sementara.
GUNAKAN_GOOGLE_DRIVE = True

if DI_GOOGLE_COLAB and GUNAKAN_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    FOLDER_DATABASE = Path("/content/drive/MyDrive/StudentDatabase")
elif DI_GOOGLE_COLAB:
    FOLDER_DATABASE = Path("/content/StudentDatabase")
else:
    # Saat dijalankan di komputer, folder dibuat di lokasi kerja saat ini.
    FOLDER_DATABASE = Path.cwd() / "StudentDatabase"

FOLDER_DATABASE.mkdir(parents=True, exist_ok=True)
DB_PATH = FOLDER_DATABASE / "kampus.db"

print(f"Lokasi database: {DB_PATH}")


# ============================================================
# 2. KONEKSI DATABASE
# ============================================================

@contextmanager
def koneksi_database():
    """Membuka koneksi, menyimpan transaksi, lalu menutup koneksi."""
    conn = sqlite3.connect(str(DB_PATH), timeout=10)
    try:
        conn.execute("PRAGMA foreign_keys = ON")
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


def buat_database():
    """Membuat database dan tabel secara otomatis jika belum tersedia."""
    try:
        with koneksi_database() as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS mahasiswa (
                    nim TEXT PRIMARY KEY CHECK (length(trim(nim)) > 0),
                    nama TEXT NOT NULL CHECK (length(trim(nama)) > 0),
                    prodi TEXT NOT NULL CHECK (length(trim(prodi)) > 0),
                    ipk REAL NOT NULL CHECK (ipk >= 0 AND ipk <= 4)
                )
            """)
        print("Database dan tabel mahasiswa siap digunakan.")
    except sqlite3.Error as e:
        print(f"Gagal menyiapkan database: {e}")
        raise


# ============================================================
# 3. VALIDASI INPUT
# ============================================================

def input_ipk(pesan="Masukkan IPK (0–4): "):
    """Meminta IPK sampai input berupa angka dalam rentang 0 sampai 4."""
    while True:
        teks = input(pesan).strip().replace(",", ".")
        try:
            ipk = float(teks)
        except ValueError:
            print("Input tidak valid. IPK harus berupa angka, misalnya 3.50.")
            continue

        if 0 <= ipk <= 4:
            return ipk

        print("IPK harus berada pada rentang 0 sampai 4.")


def input_teks_wajib(pesan):
    """Meminta teks yang tidak boleh kosong."""
    while True:
        nilai = input(pesan).strip()
        if nilai:
            return nilai
        print("Input tidak boleh kosong. Silakan coba lagi.")


# ============================================================
# 4. CREATE - TAMBAH MAHASISWA
# ============================================================

def tambah_mahasiswa():
    print("\n=== TAMBAH MAHASISWA ===")
    nim = input_teks_wajib("Masukkan NIM: ")
    nama = input_teks_wajib("Masukkan nama: ")
    prodi = input_teks_wajib("Masukkan program studi: ")
    ipk = input_ipk()

    try:
        with koneksi_database() as conn:
            conn.execute("""
                INSERT INTO mahasiswa (nim, nama, prodi, ipk)
                VALUES (?, ?, ?, ?)
            """, (nim, nama, prodi, ipk))
        print("Mahasiswa berhasil ditambahkan.")
    except sqlite3.IntegrityError:
        print("Gagal: NIM sudah terdaftar atau data melanggar aturan database.")
    except sqlite3.Error as e:
        print(f"Terjadi kesalahan database: {e}")


# ============================================================
# 5. READ - LIHAT SEMUA MAHASISWA
# ============================================================

def lihat_mahasiswa():
    print("\n=== DAFTAR MAHASISWA ===")
    try:
        with koneksi_database() as conn:
            data = conn.execute("""
                SELECT nim, nama, prodi, ipk
                FROM mahasiswa
                ORDER BY nim
            """).fetchall()

        if not data:
            print("Belum ada data mahasiswa.")
            return

        garis = "-" * 86
        print(garis)
        print(f"{'NIM':<16} {'NAMA':<25} {'PROGRAM STUDI':<30} {'IPK':>6}")
        print(garis)
        for nim, nama, prodi, ipk in data:
            print(f"{nim:<16} {nama[:25]:<25} {prodi[:30]:<30} {ipk:>6.2f}")
        print(garis)
        print(f"Jumlah data: {len(data)} mahasiswa")
    except sqlite3.Error as e:
        print(f"Gagal membaca database: {e}")


# ============================================================
# 6. READ - CARI MAHASISWA
# ============================================================

def cari_mahasiswa():
    print("\n=== CARI MAHASISWA ===")
    print("1. Cari berdasarkan NIM")
    print("2. Cari berdasarkan nama")
    pilihan = input("Pilih metode pencarian (1/2): ").strip()

    if pilihan not in ("1", "2"):
        print("Pilihan tidak valid. Masukkan 1 atau 2.")
        return

    kata_kunci = input_teks_wajib("Masukkan kata kunci: ")

    try:
        with koneksi_database() as conn:
            if pilihan == "1":
                data = conn.execute("""
                    SELECT nim, nama, prodi, ipk
                    FROM mahasiswa
                    WHERE nim = ?
                """, (kata_kunci,)).fetchall()
            else:
                data = conn.execute("""
                    SELECT nim, nama, prodi, ipk
                    FROM mahasiswa
                    WHERE nama LIKE ? COLLATE NOCASE
                    ORDER BY nama
                """, (f"%{kata_kunci}%",)).fetchall()

        if not data:
            print("Mahasiswa tidak ditemukan.")
            return

        print("\nHasil pencarian:")
        for nim, nama, prodi, ipk in data:
            print("-" * 35)
            print(f"NIM  : {nim}")
            print(f"Nama : {nama}")
            print(f"Prodi: {prodi}")
            print(f"IPK  : {ipk:.2f}")
        print("-" * 35)
    except sqlite3.Error as e:
        print(f"Gagal mencari data: {e}")


# ============================================================
# 7. UPDATE - MEMPERBARUI IPK
# ============================================================

def update_ipk():
    print("\n=== UPDATE IPK ===")
    nim = input_teks_wajib("Masukkan NIM mahasiswa: ")
    ipk_baru = input_ipk("Masukkan IPK baru (0–4): ")

    try:
        with koneksi_database() as conn:
            cursor = conn.execute("""
                UPDATE mahasiswa
                SET ipk = ?
                WHERE nim = ?
            """, (ipk_baru, nim))
            jumlah_diubah = cursor.rowcount

        if jumlah_diubah == 0:
            print("NIM tidak ditemukan. Tidak ada data yang diubah.")
        else:
            print(f"IPK mahasiswa {nim} berhasil diperbarui menjadi {ipk_baru:.2f}.")
    except sqlite3.Error as e:
        print(f"Gagal memperbarui IPK: {e}")


# ============================================================
# 8. DELETE - HAPUS MAHASISWA
# ============================================================

def hapus_mahasiswa():
    print("\n=== HAPUS MAHASISWA ===")
    nim = input_teks_wajib("Masukkan NIM mahasiswa yang akan dihapus: ")

    try:
        with koneksi_database() as conn:
            mahasiswa = conn.execute("""
                SELECT nama FROM mahasiswa
                WHERE nim = ?
            """, (nim,)).fetchone()

        if mahasiswa is None:
            print("Mahasiswa tidak ditemukan.")
            return

        konfirmasi = input(
            f"Yakin menghapus {mahasiswa[0]} (NIM {nim})? Ketik y untuk lanjut: "
        ).strip().lower()

        if konfirmasi != "y":
            print("Penghapusan dibatalkan.")
            return

        with koneksi_database() as conn:
            cursor = conn.execute(
                "DELETE FROM mahasiswa WHERE nim = ?", (nim,)
            )
            jumlah_dihapus = cursor.rowcount

        if jumlah_dihapus:
            print("Mahasiswa berhasil dihapus.")
        else:
            print("Data sudah tidak tersedia; tidak ada yang dihapus.")
    except sqlite3.Error as e:
        print(f"Gagal menghapus data: {e}")


# ============================================================
# 9. ANALISIS IPK
# ============================================================

def analisis_ipk():
    print("\n=== ANALISIS IPK ===")
    try:
        with koneksi_database() as conn:
            jumlah, rata_rata, tertinggi, terendah = conn.execute("""
                SELECT COUNT(*), AVG(ipk), MAX(ipk), MIN(ipk)
                FROM mahasiswa
            """).fetchone()

            data_per_prodi = conn.execute("""
                SELECT prodi, COUNT(*), AVG(ipk)
                FROM mahasiswa
                GROUP BY prodi
                ORDER BY AVG(ipk) DESC
            """).fetchall()

        print(f"Jumlah mahasiswa : {jumlah}")
        if jumlah == 0:
            print("Belum ada data untuk dianalisis.")
            return

        print(f"Rata-rata IPK    : {rata_rata:.2f}")
        print(f"IPK tertinggi    : {tertinggi:.2f}")
        print(f"IPK terendah     : {terendah:.2f}")

        print("\n=== RATA-RATA IPK PER PRODI ===")
        for prodi, jumlah_prodi, rata_prodi in data_per_prodi:
            print(
                f"{prodi}: {jumlah_prodi} mahasiswa, "
                f"rata-rata IPK {rata_prodi:.2f}"
            )
    except sqlite3.Error as e:
        print(f"Gagal melakukan analisis: {e}")


# ============================================================
# 10. DATA DEMO OPSIONAL
# ============================================================

def isi_data_demo_jika_kosong():
    """Memasukkan lima data contoh hanya jika tabel masih kosong."""
    data_demo = [
        ("260001", "Andi Saputra", "Teknologi Informasi", 3.50),
        ("260002", "Budi Santoso", "Teknologi Informasi", 3.75),
        ("260003", "Citra Lestari", "Sistem Informasi", 3.90),
        ("260004", "Dina Permata", "Sistem Informasi", 3.20),
        ("260005", "Eko Pratama", "Teknologi Informasi", 3.40),
    ]

    try:
        with koneksi_database() as conn:
            jumlah = conn.execute(
                "SELECT COUNT(*) FROM mahasiswa"
            ).fetchone()[0]

            if jumlah > 0:
                print("Tabel sudah berisi data; data demo tidak ditambahkan.")
                return

            conn.executemany("""
                INSERT INTO mahasiswa (nim, nama, prodi, ipk)
                VALUES (?, ?, ?, ?)
            """, data_demo)
        print("Lima data demo berhasil ditambahkan.")
    except sqlite3.Error as e:
        print(f"Gagal menambahkan data demo: {e}")


# ============================================================
# 11. MENU UTAMA
# ============================================================

def menu_utama():
    while True:
        print("\n" + "=" * 38)
        print("        STUDENT DATABASE")
        print("=" * 38)
        print("1. Tambah mahasiswa")
        print("2. Lihat mahasiswa")
        print("3. Cari mahasiswa")
        print("4. Update IPK")
        print("5. Hapus mahasiswa")
        print("6. Analisis IPK")
        print("7. Keluar")
        print("=" * 38)

        try:
            pilihan = input("Pilih menu (1-7): ").strip()

            if pilihan == "1":
                tambah_mahasiswa()
            elif pilihan == "2":
                lihat_mahasiswa()
            elif pilihan == "3":
                cari_mahasiswa()
            elif pilihan == "4":
                update_ipk()
            elif pilihan == "5":
                hapus_mahasiswa()
            elif pilihan == "6":
                analisis_ipk()
            elif pilihan == "7":
                print("Terima kasih. Program selesai.")
                break
            else:
                print("Pilihan tidak valid. Masukkan angka 1 sampai 7.")

        except (KeyboardInterrupt, EOFError):
            print("\nProgram dihentikan oleh pengguna.")
            break


# ============================================================
# 12. JALANKAN APLIKASI
# ============================================================

if __name__ == "__main__":
    # Jika ingin data contoh, ubah False menjadi True.
    ISI_DATA_CONTOH_SEBELUM_MENU = True

    buat_database()
    if ISI_DATA_CONTOH_SEBELUM_MENU:
        isi_data_demo_jika_kosong()

    menu_utama()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Lokasi database: /content/drive/MyDrive/StudentDatabase/kampus.db
Database dan tabel mahasiswa siap digunakan.
Lima data demo berhasil ditambahkan.

        STUDENT DATABASE
1. Tambah mahasiswa
2. Lihat mahasiswa
3. Cari mahasiswa
4. Update IPK
5. Hapus mahasiswa
6. Analisis IPK
7. Keluar
Pilih menu (1-7): 2

=== DAFTAR MAHASISWA ===
--------------------------------------------------------------------------------------
NIM              NAMA                      PROGRAM STUDI                     IPK
--------------------------------------------------------------------------------------
260001           Andi Saputra              Teknologi Informasi              3.50
260002           Budi Santoso              Teknologi Informasi              3.75
260003           Citra Lestari             Sistem Informasi                 3.90
260004           Dina Permata        